# EXPERIMENT 4B – Clustering Analysis using K-Medoids on Credit Card Dataset

**Aim:** To implement the K-Medoids clustering algorithm on the Credit Card dataset and determine the optimum number of clusters using the Silhouette Method.

**Dataset:** [Credit Card Dataset – Kaggle](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)

**Algorithm:** K-Medoids (PAM-style alternate update).

**Important:** K-Medoids is based on distances between observations and actual data points (medoids). To keep this practical easy to run in a standard laptop environment, this notebook uses a reproducible sample of up to 2,000 customer records. The method and silhouette selection are demonstrated on this sample.

## 1. Import required libraries
We use Pandas and NumPy for data handling, Matplotlib for plotting, and Scikit-learn for preprocessing, distances and silhouette score.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import preprocessing, distance and evaluation tools
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances, silhouette_score

## 2. Load the Credit Card dataset

Download `CC GENERAL.csv` from Kaggle and place it in the same folder as this notebook. If needed, change the file path below.

In [ ]:
# Load the credit card customer dataset
df = pd.read_csv("CC GENERAL.csv")

# Display initial records and dataset information
display(df.head())
print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

## 3. Data preprocessing
`CUST_ID` is an identifier, so it is removed. The remaining customer attributes are converted to numeric values, and missing values are filled using each column's median.

In [ ]:
# Remove customer ID because it is not a behavioural feature
data = df.drop(columns=["CUST_ID"], errors="ignore").copy()

# Convert all selected columns to numeric values
data = data.apply(pd.to_numeric, errors="coerce")

# Replace infinite values with missing values
data = data.replace([np.inf, -np.inf], np.nan)

# Fill missing values using each column's median
data = data.fillna(data.median())

# Remove any columns that contain no usable values
data = data.dropna(axis=1, how="all")

print("Features used:", data.columns.tolist())
print("Preprocessed shape:", data.shape)
print("Remaining missing values:", int(data.isnull().sum().sum()))

## 4. Sample records and scale the features
The full dataset has thousands of records. K-Medoids requires many pairwise distance calculations, so we take a random sample of up to 2,000 records to keep execution time and memory manageable. Standardization prevents high-range features from dominating distance calculations.

In [ ]:
# Use a reproducible sample so results can be repeated
sample_size = min(2000, len(data))
data_sample = data.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Standardize all selected customer features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_sample)

print("Number of sampled customers:", X_scaled.shape[0])
print("Number of features:", X_scaled.shape[1])

## 5. Create a K-Medoids algorithm

A medoid is an actual observation that represents a cluster. The algorithm works as follows:

1. Select K observations as initial medoids.
2. Assign each observation to its nearest medoid.
3. In each cluster, select the observation with the smallest total distance to the other cluster members.
4. Repeat assignment and medoid updates until the medoids stop changing or the maximum number of iterations is reached.

This is an educational alternate-update implementation of K-Medoids.

In [ ]:
def k_medoids(distance_matrix, k, max_iter=100, random_state=42):
    # Create a random generator so initialization is reproducible
    rng = np.random.default_rng(random_state)
    n_samples = distance_matrix.shape[0]

    # Choose K different observations as the starting medoids
    medoids = rng.choice(n_samples, size=k, replace=False)

    for iteration in range(max_iter):
        # Calculate distance from every point to each current medoid
        distances_to_medoids = distance_matrix[:, medoids]

        # Assign every point to its nearest medoid
        labels = np.argmin(distances_to_medoids, axis=1)

        # Store the updated medoid indices
        new_medoids = medoids.copy()

        # Find the best medoid for each cluster
        for cluster_id in range(k):
            members = np.where(labels == cluster_id)[0]

            # Keep the old medoid if a cluster has no members
            if len(members) == 0:
                continue

            # Distances between all members of this cluster
            within_cluster_distances = distance_matrix[np.ix_(members, members)]

            # Total distance from each candidate to all cluster members
            total_distances = within_cluster_distances.sum(axis=1)

            # The member with minimum total distance becomes the medoid
            new_medoids[cluster_id] = members[np.argmin(total_distances)]

        # Stop when medoids no longer change
        if np.array_equal(new_medoids, medoids):
            break

        medoids = new_medoids

    # Reassign points using the final medoids
    final_distances = distance_matrix[:, medoids]
    final_labels = np.argmin(final_distances, axis=1)

    return final_labels, medoids

## 6. Calculate the distance matrix
We use Euclidean distance between every pair of scaled customer records. The resulting matrix is reused for each tested K value.

In [ ]:
# Calculate pairwise Euclidean distances between sampled customers
distance_matrix = pairwise_distances(X_scaled, metric="euclidean")

print("Distance matrix shape:", distance_matrix.shape)

## 7. Determine the number of clusters using the Silhouette Method

We test different values of K. The silhouette score measures how close each point is to its own cluster compared with other clusters. A higher score generally indicates better-defined clusters.

We test K from 2 to 6. K must be at least 2 for silhouette score to be defined.

In [ ]:
# Store the silhouette score for each tested K
silhouette_scores = {}
k_values = range(2, 7)

for k in k_values:
    # Run K-Medoids for the current number of clusters
    labels, medoids = k_medoids(
        distance_matrix,
        k=k,
        max_iter=100,
        random_state=42
    )

    # Calculate silhouette score using the precomputed distances
    score = silhouette_score(
        distance_matrix,
        labels,
        metric="precomputed"
    )

    silhouette_scores[k] = score
    print(f"K = {k}, Silhouette Score = {score:.4f}")

## 8. Plot the Silhouette scores and choose the best K
The K value with the highest silhouette score among those tested is selected.

In [ ]:
# Find K with the highest silhouette score
optimal_k = max(silhouette_scores, key=silhouette_scores.get)

# Plot silhouette score against number of clusters
plt.figure(figsize=(7, 4))
plt.plot(
    list(silhouette_scores.keys()),
    list(silhouette_scores.values()),
    marker="o"
)
plt.title("Silhouette Method for K-Medoids")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(silhouette_scores.keys()))
plt.grid(True)
plt.show()

print("Optimal number of clusters:", optimal_k)

## 9. Fit the final K-Medoids model
Run the algorithm again using the selected K and attach the cluster labels to the sampled customer records.

In [ ]:
# Train final K-Medoids model using the selected K
final_labels, final_medoids = k_medoids(
    distance_matrix,
    k=optimal_k,
    max_iter=100,
    random_state=42
)

# Add cluster assignments to the sampled dataset
clustered_data = data_sample.copy()
clustered_data["Cluster"] = final_labels

print("Final K-Medoids clustering completed.")
print("\nCustomers in each cluster:")
print(clustered_data["Cluster"].value_counts().sort_index())

print("\nMedoid row indices in the sample:", final_medoids)

## 10. Display cluster summary
The mean of each feature is calculated for every cluster. This helps describe the customer groups.

In [ ]:
# Calculate average feature values for each cluster
cluster_summary = clustered_data.groupby("Cluster").mean()

display(cluster_summary)

## 11. Conclusion

K-Medoids clustering was implemented on the Credit Card dataset. The data was cleaned, standardized and sampled to make pairwise distance calculations practical. The Silhouette Method was used to compare different values of K, and the K with the highest silhouette score in the tested range was selected.

**Result:** The silhouette scores, selected number of clusters and cluster summary are displayed when the notebook is executed.

**Practical note:** This notebook implements K-Medoids directly using NumPy and Scikit-learn distance utilities, so it does not require installing the extra `scikit-learn-extra` package.